# Three-Panel Plot: Skymodel | CASA Observation | Fit Residual
Plots zoomed versions of (1) the input skymodel, (2) the CASA pbcor image, and (3) the imfit residual image side by side.

## Configuration

In [1]:
# ── Paths ──────────────────────────────────────────────────────────────────────
PBCOR_PATH     = r'C:\Users\nachi\Documents\JupyterNotebooksLinux\Cosmic Origins\casa_main\pbcor_imgs'
SKYMODEL_PATH  = r'C:\Users\nachi\Documents\JupyterNotebooksLinux\Cosmic Origins\casa_main\skymodels'
RESIDUAL_PATH  = r'C:\Users\nachi\Documents\JupyterNotebooksLinux\Cosmic Origins\casa_main\residual_imgs'  # folder where residual FITS files are saved
RESULTS_PATH   = r'C:\Users\nachi\Documents\JupyterNotebooksLinux\Cosmic Origins\casa_main\fitting_results.json'
MODEL_PATH     = r'C:\Users\nachi\Documents\JupyterNotebooksLinux\Cosmic Origins\casa_main\model_imgs'  # folder where model FITS files are saved
OUT_DIR        = 'plots_three_panel'

# ── Display ────────────────────────────────────────────────────────────────────
CMAP      = 'jet'
LOG_SCALE = False
VMIN_PCT  = 0
VMAX_PCT  = 99.5
DPI       = 130

## Imports

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from matplotlib.colors import LogNorm, TwoSlopeNorm
from astropy.io import fits

from plotting_utils import (
    DISTANCES_PC,
    get_pixel_scale_arcsec,
    make_norm,
    make_residual_norm,
    zoom_bounds,
    add_AU_ticks,
    draw_ellipse_on_ax,
    style_ax,
    add_colorbar,
    plot_three_panel,
)

%matplotlib inline

if OUT_DIR:
    os.makedirs(OUT_DIR, exist_ok=True)


## Load fitting results

In [3]:
with open(RESULTS_PATH, 'r') as f:
    master_dict = json.load(f)

field_names = ['Orion', 'Perseus']
print(f'Loaded results for {len(master_dict)} snapshot(s):')
for snap, fields in master_dict.items():
    for field, axes in fields.items():
        if field not in field_names:
            continue
        for axis in axes:
            print(f'  snapshot {snap} | {field} | axis {axis}')

Loaded results for 23 snapshot(s):
  snapshot 170 | Orion | axis x
  snapshot 170 | Orion | axis y
  snapshot 170 | Orion | axis z
  snapshot 170 | Perseus | axis x
  snapshot 170 | Perseus | axis y
  snapshot 170 | Perseus | axis z
  snapshot 171 | Orion | axis x
  snapshot 171 | Orion | axis y
  snapshot 171 | Orion | axis z
  snapshot 171 | Perseus | axis x
  snapshot 171 | Perseus | axis y
  snapshot 171 | Perseus | axis z
  snapshot 172 | Orion | axis x
  snapshot 172 | Orion | axis y
  snapshot 172 | Orion | axis z
  snapshot 172 | Perseus | axis x
  snapshot 172 | Perseus | axis y
  snapshot 172 | Perseus | axis z
  snapshot 173 | Orion | axis x
  snapshot 173 | Orion | axis y
  snapshot 173 | Orion | axis z
  snapshot 173 | Perseus | axis x
  snapshot 173 | Perseus | axis y
  snapshot 173 | Perseus | axis z
  snapshot 174 | Orion | axis x
  snapshot 174 | Orion | axis y
  snapshot 174 | Orion | axis z
  snapshot 174 | Perseus | axis x
  snapshot 174 | Perseus | axis y
  snapsho

In [4]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import h5py
import yt
import os
import json
%matplotlib inline

def load_dict(path):
    if os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    print(f"No file found at {path}, starting fresh.")
    return {}

def save_dict(d, path):
    with open(path, 'w') as f:
        json.dump(d, f, indent=2)
    print(f"Saved to {path}")

fitting_path = r'C:\Users\nachi\Documents\JupyterNotebooksLinux\Cosmic Origins\casa_main\fitting_results.json'
master_dict = load_dict(fitting_path) 

mass_path = "mass_results.json"
mass_dict = load_dict(mass_path)

snapshots = master_dict.keys()
field_names = ["Orion","Perseus"]
rows = []
for snapshot in master_dict.keys():
    print(f"{snapshot}")
    snapshot_time = master_dict[snapshot]["time"]
    try:
        masses = mass_dict[f'snapshot_{snapshot}']
    except KeyError:
        print(f"No mass data found for snapshot {snapshot}")
        continue
    for field in master_dict[snapshot].keys():
        if field not in field_names:
            continue
        for axis in master_dict[snapshot][field].keys():
            d = master_dict[snapshot][field][axis]
            rows.append({
                'snapshot':          snapshot,
                'axis':              axis,
                'field':             field,
                'inclination_deg':   d['inc'],
                'mass_fit_Msun':     d['dust_mass_Msun'],
                'true_mass_1e7':     masses[1],
                'true_mass_1e8':     masses[2],
                'true_mass_1e9':     masses[3],
                'pct_diff_1e8':    100 * (masses[2] - d['dust_mass_Msun']) / masses[2],
                'pct_diff_1e9':    100 * (masses[3] - d['dust_mass_Msun']) / masses[3],
                'fitting_radius_AU':    d['radius_AU'],
                'fitting_radius_Tobin':     d['radius_AU_Tobin'],
                'snr': d['snr'],
                "peak_residual_fraction": d['peak_residual_fraction'],
                "time_Myr": snapshot_time
            })

df = pd.DataFrame(rows)


170
171
172
173
174
175
180
118
315
325
327
No mass data found for snapshot 327
330
334
343
386
389
408
160
161
162
163
164
165


## Helper functions

In [ ]:
# get_pixel_scale_arcsec, make_norm, make_residual_norm, zoom_bounds, add_AU_ticks,
# draw_ellipse_on_ax, style_ax and add_colorbar now live in plotting_utils.py (imported
# above) so the CLI script (plot_three_panel.py) and this notebook share one implementation.


## Three-panel plot function

In [ ]:
# plot_three_panel() now lives in plotting_utils.py (imported above) and is shared
# with plot_three_panel.py, the batch CLI driver.


In [7]:
plt.rcParams.update({
    'axes.labelsize': 11,
    'axes.titlesize': 11,
    'xtick.labelsize': 30,
    'ytick.labelsize': 30,
    'legend.fontsize': 3,
    'figure.titlesize': 11
})

## Plot a single snapshot

In [ ]:
SNAPSHOT    = '172'
FIELD       = 'Orion'
AXIS        = 'z'
ZOOM_FACTOR = 3

pbcor_fname = f'ALMA_snapshot_{SNAPSHOT}_axis_{AXIS}_{FIELD}_sim_observed_pbcor.fits'
pbcor_fpath = os.path.join(PBCOR_PATH, pbcor_fname)
fit         = master_dict[SNAPSHOT][FIELD][AXIS]

plot_three_panel(pbcor_fpath, fit, SKYMODEL_PATH, RESIDUAL_PATH,
                  zoom_factor=ZOOM_FACTOR, cmap=CMAP, dpi=DPI, out_dir=OUT_DIR, show=True)


In [ ]:
def plot_three_panel_stack(snapshots, field, axis, zoom_factor=3, save=True, mass_dict=None, df=None):
    """
    Stack multiple snapshots as rows of three panels each:
      [0] Skymodel (zoomed)
      [1] CASA pbcor observation (zoomed)
      [2] imfit residual image (zoomed)

    Parameters
    ----------
    snapshots   : list of str — snapshot numbers e.g. ['170','171',...,'175']
    field       : str — 'Orion' or 'Perseus'
    axis        : str — 'x', 'y', or 'z'
    zoom_factor : int — half-width of zoom in units of Rmaj
    save        : bool — whether to save PNG to OUT_DIR
    """

    n_rows = len(snapshots)
    fig, axes = plt.subplots(n_rows, 3, figsize=(24, 10 * n_rows),
                             gridspec_kw={'wspace': 0.35, 'hspace': 0.4})
    fig.patch.set_facecolor('#0d0d0d')
    #fig.suptitle(f'Snapshots {snapshots[0]}–{snapshots[-1]}  |  {field}  |  axis {axis}',
    #             color='white', y=1.01)

    # Ensure axes is always 2D even if n_rows == 1
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    for row_idx, snapshot in enumerate(snapshots):

        ax_sky, ax_obs, ax_res = axes[row_idx]
        for ax in [ax_sky, ax_obs, ax_res]:
            style_ax(ax)

        distance_pc = DISTANCES_PC.get(field, 400)

        # ── Load pbcor ────────────────────────────────────────────────────────
        pbcor_fname = f'ALMA_snapshot_{snapshot}_axis_{axis}_{field}_sim_observed_pbcor.fits'
        pbcor_fpath = os.path.join(PBCOR_PATH, pbcor_fname)
        if not os.path.exists(pbcor_fpath):
            print(f'[skip] pbcor not found: {pbcor_fname}')
            for ax in [ax_sky, ax_obs, ax_res]:
                ax.text(0.5, 0.5, f'snap {snapshot}\nnot found',
                        transform=ax.transAxes, color='red',
                        ha='center', va='center')
            continue
        with fits.open(pbcor_fpath) as hdul:
            pb_hdr  = hdul[0].header
            pb_data = hdul[0].data.squeeze()
        pb_ny, pb_nx = pb_data.shape
        pb_pix_as    = get_pixel_scale_arcsec(pb_hdr)
        pb_pix_AU    = pb_pix_as * distance_pc
        pb_cx, pb_cy = pb_nx / 2.0, pb_ny / 2.0

        # ── Load skymodel ─────────────────────────────────────────────────────
        sky_fname = f'snapshot_{snapshot}_{field}_flux_map_ALMA_axis_{axis}.fits'
        sky_fpath = os.path.join(SKYMODEL_PATH, sky_fname)
        if not os.path.exists(sky_fpath):
            print(f'[skip] skymodel not found: {sky_fname}')
            continue
        with fits.open(sky_fpath) as hdul:
            sky_hdr  = hdul[0].header
            sky_data = hdul[0].data.squeeze()
        sky_ny, sky_nx = sky_data.shape
        sky_pix_as     = get_pixel_scale_arcsec(sky_hdr)
        sky_pix_AU     = sky_pix_as * distance_pc
        sky_cx, sky_cy = sky_nx / 2.0, sky_ny / 2.0

        # ── Load residual ─────────────────────────────────────────────────────
        res_fname = f'ALMA_snapshot_{snapshot}_axis_{axis}_{field}_sim_observed_pbcor_residual.fits'
        res_fpath = os.path.join(RESIDUAL_PATH, res_fname)
        if not os.path.exists(res_fpath):
            print(f'[skip] residual not found: {res_fname}')
            continue
        with fits.open(res_fpath) as hdul:
            res_hdr  = hdul[0].header
            res_data = hdul[0].data.squeeze()
        res_ny, res_nx = res_data.shape
        res_pix_as     = get_pixel_scale_arcsec(res_hdr)
        res_cx, res_cy = res_nx / 2.0, res_ny / 2.0

        # ── Fit parameters ────────────────────────────────────────────────────
        fit = master_dict.get(snapshot, {}).get(field, {}).get(axis, {})
        Rmaj_as  = fit.get('Rmaj')
        Rmin_as  = fit.get('Rmin')
        pa_deg   = fit.get('pa')
        inc      = fit.get('inc')
        r_AU_T   = fit.get('radius_AU_Tobin')
        flux     = fit.get('flux')
        snr      = fit.get('snr')
        prf      = fit.get('peak_residual_fraction')
        peak_res = fit.get('peak_residual')

        fit_valid = all(v is not None and not (isinstance(v, float) and np.isnan(v))
                        for v in [Rmaj_as, Rmin_as, pa_deg])

        if fit_valid:
            Rmaj_AU      = Rmaj_as * distance_pc
            Rmin_AU      = Rmin_as * distance_pc
            pb_Rmaj_pix  = Rmaj_as / pb_pix_as
            pb_Rmin_pix  = Rmin_as / pb_pix_as
            sk_Rmaj_pix  = Rmaj_as / sky_pix_as
            sk_Rmin_pix  = Rmin_as / sky_pix_as
            res_Rmaj_pix = Rmaj_as / res_pix_as
            res_Rmin_pix = Rmin_as / res_pix_as
            mpl_angle    = 90 + pa_deg
        else:
            print(f'[warn] NaN fit: snap {snapshot} | {field} | axis {axis}')
            pb_Rmaj_pix  = pb_nx * 0.1
            sk_Rmaj_pix  = sky_nx * 0.1
            res_Rmaj_pix = res_nx * 0.1
            mpl_angle    = 0

        # ── Mass lookup ───────────────────────────────────────────────────────────
        fitted_mass = np.nan
        true_mass   = np.nan

        if df is not None:
            row = df[(df['snapshot'] == snapshot) & 
                    (df['field'] == field) & 
                    (df['axis'] == axis)]
            if not row.empty:
                fitted_mass = row['mass_fit_Msun'].values[0]

        if mass_dict is not None:
            key = f'snapshot_{snapshot}'
            if key in mass_dict:
                true_mass = mass_dict[key][2]  # index 2 = 1e-8 opacity
                
        # ── Panel 0: Skymodel ─────────────────────────────────────────────────
        sx0, sx1, sy0, sy1 = zoom_bounds(sky_cx, sky_cy, sk_Rmaj_pix,
                                         sky_nx, sky_ny, factor=zoom_factor)
        szd = sky_data[sy0:sy1, sx0:sx1]
        im0 = ax_sky.imshow(szd, origin='lower', cmap=CMAP, norm=make_norm(szd, vmin_pct=VMIN_PCT, vmax_pct=VMAX_PCT, log_scale=LOG_SCALE),
                            extent=[sx0, sx1, sy0, sy1])
        if fit_valid:
            draw_ellipse_on_ax(ax_sky, sky_cx, sky_cy, sk_Rmaj_pix, sk_Rmin_pix, mpl_angle)
        ax_sky.axhline(sky_cy, color='white', lw=0.4, alpha=0.35)
        ax_sky.axvline(sky_cx, color='white', lw=0.4, alpha=0.35)
        ax_sky.set_xlim(sx0, sx1); ax_sky.set_ylim(sy0, sy1)
        add_AU_ticks(ax_sky, sky_cx, sky_cy, sx0, sx1, sy0, sy1, sky_pix_as, distance_pc)
        ax_sky.set_xlabel(f'ΔRA (AU)  [d = {distance_pc} pc]', color='white')
        ax_sky.set_ylabel('ΔDec (AU)', color='white')
        ax_sky.set_title(f'snap {snapshot} — Skymodel', color='white')
        add_colorbar(fig, im0, ax_sky, sky_hdr.get('BUNIT', 'Jy/pixel'))

        # ── Panel 1: CASA pbcor ───────────────────────────────────────────────
        px0, px1, py0, py1 = zoom_bounds(pb_cx, pb_cy, pb_Rmaj_pix,
                                         pb_nx, pb_ny, factor=zoom_factor)
        pzd = pb_data[py0:py1, px0:px1]
        im1 = ax_obs.imshow(pzd, origin='lower', cmap=CMAP, norm=make_norm(pzd, vmin_pct=VMIN_PCT, vmax_pct=VMAX_PCT, log_scale=LOG_SCALE),
                            extent=[px0, px1, py0, py1])
        if fit_valid:
            draw_ellipse_on_ax(ax_obs, pb_cx, pb_cy, pb_Rmaj_pix, pb_Rmin_pix, mpl_angle)
        ax_obs.axhline(pb_cy, color='white', lw=0.4, alpha=0.35)
        ax_obs.axvline(pb_cx, color='white', lw=0.4, alpha=0.35)
        ax_obs.set_xlim(px0, px1); ax_obs.set_ylim(py0, py1)
        add_AU_ticks(ax_obs, pb_cx, pb_cy, px0, px1, py0, py1, pb_pix_as, distance_pc)
        ax_obs.set_xlabel(f'ΔRA (AU)  [d = {distance_pc} pc]', color='white')
        ax_obs.set_ylabel('ΔDec (AU)', color='white')
        ax_obs.set_title(f'snap {snapshot} — CASA Observation', color='white')
        add_colorbar(fig, im1, ax_obs, pb_hdr.get('BUNIT', 'Jy/beam'))

        # Annotation box
        if fit_valid:
            info = (
                f"Rmaj = {Rmaj_as:.3f}\"  ({Rmaj_AU/2:.0f} AU)\n"
                f"Rmin = {Rmin_as:.3f}\"  ({Rmin_AU/2:.0f} AU)\n"
                f"PA = {pa_deg:.1f}°    i = {inc:.1f}°\n"
                f"Flux = {flux:.3e} Jy\n"
                f"R_disk (Tobin) = {r_AU_T:.1f} AU\n"
                f"snr = {snr:.1f}\n"
                f"peak residual = {peak_res:.3e} Jy  ({prf:.1%} of peak)\n"
                f"─────────────────────────\n"
                f"Fitted mass  = {fitted_mass:.3e} M☉\n"
                f"True mass    = {true_mass:.3e} M☉  (κ=1e-8)\n"
                f"Ratio fit/true = {fitted_mass/true_mass:.2f}" if not np.isnan(fitted_mass) and not np.isnan(true_mass) and true_mass != 0
                else f"Fitted mass  = {fitted_mass:.3e} M☉\nTrue mass    = N/A"
            )
            color, edge = 'white', 'cyan'
        else:
            info = 'Fit failed — no Gaussian parameters'
            color, edge = 'orange', 'orange'
        ax_obs.text(0.02, 0.98, info, transform=ax_obs.transAxes,
                    va='top', ha='left', color=color, family='monospace',
                    bbox=dict(boxstyle='round,pad=0.4', facecolor='#111',
                              edgecolor=edge, alpha=0.88))

        # ── Panel 2: Residual ─────────────────────────────────────────────────
        rx0, rx1, ry0, ry1 = zoom_bounds(res_cx, res_cy, res_Rmaj_pix,
                                         res_nx, res_ny, factor=zoom_factor)
        rzd = res_data[ry0:ry1, rx0:rx1]
        im2 = ax_res.imshow(rzd, origin='lower', cmap='RdBu_r',
                            norm=make_residual_norm(rzd),
                            extent=[rx0, rx1, ry0, ry1])
        if fit_valid:
            draw_ellipse_on_ax(ax_res, res_cx, res_cy, res_Rmaj_pix, res_Rmin_pix,
                               mpl_angle, color='black', ls='--')
        ax_res.axhline(res_cy, color='gray', lw=0.4, alpha=0.35)
        ax_res.axvline(res_cx, color='gray', lw=0.4, alpha=0.35)
        ax_res.set_xlim(rx0, rx1); ax_res.set_ylim(ry0, ry1)
        add_AU_ticks(ax_res, res_cx, res_cy, rx0, rx1, ry0, ry1, res_pix_as, distance_pc)
        ax_res.set_xlabel(f'ΔRA (AU)  [d = {distance_pc} pc]', color='white')
        ax_res.set_ylabel('ΔDec (AU)', color='white')
        ax_res.set_title(f'snap {snapshot} — imfit Residual', color='white')
        add_colorbar(fig, im2, ax_res, 'Jy/beam  (obs − model)')

    # ── Save / show ───────────────────────────────────────────────────────────
    plt.tight_layout()
    if save and OUT_DIR:
        out_fname = f'stack_{snapshots[0]}_{snapshots[-1]}_{field}_axis{axis}.png'
        fig.savefig(os.path.join(OUT_DIR, out_fname), dpi=DPI,
                    bbox_inches='tight', facecolor=fig.get_facecolor())
        print(f'Saved: {out_fname}')
    plt.show()
    plt.close(fig)


# ── Function Call ───────────────────────────────────────────────────────────────────
plot_three_panel_stack(
    snapshots=[str(s) for s in range(170, 176)],
    field='Orion',
    axis='x',
    zoom_factor=3,
    mass_dict=mass_dict,
    df=df
)

In [ ]:
from astropy.io import fits
import numpy as np

def model_loader(snapshot,axis,field):
    model_fname = f'ALMA_snapshot_{snapshot}_axis_{axis}_{field}_sim_observed_pbcor_model.fits'
    model_fpath = os.path.join(MODEL_PATH, model_fname)
    with fits.open(model_fpath) as hdul:
        model_data = hdul[0].data.squeeze()
    return model_data

    
snapshot = '170'
axis = 'y'
field = 'Perseus'
model_data = model_loader(snapshot, axis, field)


print('Model peak:', np.nanmax(model_data))
print('Model integrated (pixel sum):', np.nansum(model_data))
plt.imshow(model_data, origin='lower', cmap='plasma')
plt.colorbar(label='Jy/beam')
plt.show()

In [15]:
def get_integrated_flux_thresh(fits_path, sigma_thresh=3.0):
    """
    Same as above but only includes pixels above sigma_thresh * rms noise.
    RMS is estimated from the outer 10% of the image where signal is minimal.
    """
    with fits.open(fits_path) as hdul:
        hdr  = hdul[0].header
        data = hdul[0].data.squeeze()

    bmaj_deg = hdr['BMAJ']
    bmin_deg = hdr['BMIN']
    cdelt    = abs(hdr['CDELT1'])
    beam_area_pix = (np.pi / (4 * np.log(2))) * (bmaj_deg * bmin_deg) / (cdelt ** 2)

    # --- Estimate noise from image edges ---
    ny, nx = data.shape
    edge_fraction = 0.1
    edge_mask = np.zeros_like(data, dtype=bool)
    edge_mask[:int(ny * edge_fraction), :]  = True  # bottom
    edge_mask[int(ny * (1-edge_fraction)):, :] = True  # top
    edge_mask[:, :int(nx * edge_fraction)]  = True  # left
    edge_mask[:, int(nx * (1-edge_fraction)):] = True  # right

    rms = np.nanstd(data[edge_mask])
    threshold = sigma_thresh * rms
    print(f"RMS noise:   {rms:.4e} Jy/beam")
    print(f"Threshold:   {threshold:.4e} Jy/beam  ({sigma_thresh}σ)")

    # --- Apply threshold mask ---
    valid_mask = np.isfinite(data) & (data > threshold)
    integrated_flux_Jy = np.sum(data[valid_mask]) / beam_area_pix

    print(f"Pixels above threshold: {np.sum(valid_mask)}")
    print(f"Integrated flux:        {integrated_flux_Jy:.4e} Jy")
    return integrated_flux_Jy

fits_path = r"C:\Users\nachi\Documents\JupyterNotebooksLinux\Cosmic Origins\casa_main\pbcor_imgs\ALMA_snapshot_175_axis_y_Perseus_sim_observed_pbcor.fits"
get_integrated_flux_thresh(fits_path)

RMS noise:   1.2749e-03 Jy/beam
Threshold:   3.8248e-03 Jy/beam  (3.0σ)
Pixels above threshold: 2018
Integrated flux:        4.5073e+00 Jy


np.float64(4.5073385287956835)

In [20]:
import json
import os
import numpy as np

h  = 6.62607015e-27 #"erg*s"
c  = 2.99792458e10 #"cm/s"
kB = 1.380649e-16   #"erg/K"
nu = 3.38e11        #"Hz"

def load_dict(path):
    if os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    print(f"No file found at {path}, starting fresh.")
    return {}

def save_dict(d, path):
    with open(path, 'w') as f:
        json.dump(d, f, indent=2)
    print(f"Saved to {path}")

def Bnu(T,nu=nu):
    x = (h * nu) / (kB * T) # dimensionless variable
    return ((2 * h * nu**3 )/ (c**2)) / (np.exp(x) - 1) # erg / s / cm^2 / Hz / sr

results_path =  r'C:\Users\nachi\Documents\JupyterNotebooksLinux\Cosmic Origins\casa_main\fitting_results.json'

fitting_path = r'C:\Users\nachi\Documents\JupyterNotebooksLinux\Cosmic Origins\casa_main\fitting_results.json'
master_dict = load_dict(fitting_path) 

snapshots = master_dict.keys()
field_names = ["Orion", "Perseus"]
for snapshot in snapshots:
    if "17" not in snapshot:
        continue
    for field in master_dict[snapshot].keys():
        if field not in field_names:
            continue
        for axis in master_dict[snapshot][field].keys():

            fits_path = rf"C:\Users\nachi\Documents\JupyterNotebooksLinux\Cosmic Origins\casa_main\pbcor_imgs\ALMA_snapshot_{snapshot}_axis_{axis}_{field}_sim_observed_pbcor.fits"
            if field == "Orion":
                dist_cm = 1.234e21 # cm
            elif field == "Perseus":
                dist_cm = 9.257e20 #cm

            T_dust = 43 * (master_dict[snapshot]['total_luminosity']**0.25) 
            B_nu = Bnu(T_dust) # erg / s / cm^2 / Hz / sr

            F_v = get_integrated_flux_thresh(fits_path)
            F_v_J = F_v * 1e-23 # erg / s /cm^2 / Hz​


            dust_mass = (F_v_J * (dist_cm)**2)/(1.84 * B_nu) #g
            

            dust_mass_Msun = dust_mass / 1.989e33 # Msun
            

            master_dict[snapshot][field][axis]['dust_mass_Msun_full'] = dust_mass_Msun
            master_dict[snapshot][field][axis]['int_flux_full'] = F_v_J


            save_dict(master_dict, results_path)





            

RMS noise:   4.0002e-04 Jy/beam
Threshold:   1.2001e-03 Jy/beam  (3.0σ)
Pixels above threshold: 2566
Integrated flux:        2.1433e+00 Jy
Saved to C:\Users\nachi\Documents\JupyterNotebooksLinux\Cosmic Origins\casa_main\fitting_results.json
RMS noise:   3.5341e-04 Jy/beam
Threshold:   1.0602e-03 Jy/beam  (3.0σ)
Pixels above threshold: 2380
Integrated flux:        2.1265e+00 Jy
Saved to C:\Users\nachi\Documents\JupyterNotebooksLinux\Cosmic Origins\casa_main\fitting_results.json
RMS noise:   3.9539e-04 Jy/beam
Threshold:   1.1862e-03 Jy/beam  (3.0σ)
Pixels above threshold: 2616
Integrated flux:        2.1863e+00 Jy
Saved to C:\Users\nachi\Documents\JupyterNotebooksLinux\Cosmic Origins\casa_main\fitting_results.json
RMS noise:   7.1691e-04 Jy/beam
Threshold:   2.1507e-03 Jy/beam  (3.0σ)
Pixels above threshold: 2832
Integrated flux:        3.7047e+00 Jy
Saved to C:\Users\nachi\Documents\JupyterNotebooksLinux\Cosmic Origins\casa_main\fitting_results.json
RMS noise:   9.5406e-04 Jy/beam
Thre

## Loop over all snapshots / fields / axes

In [ ]:
for snapshot, fields in master_dict.items():
    for field, axes_dict in fields.items():
        if field not in field_names:
            continue
        for axis, fit in axes_dict.items():
            pbcor_fname = f'ALMA_snapshot_{snapshot}_axis_{axis}_{field}_sim_observed_pbcor.fits'
            pbcor_fpath = os.path.join(PBCOR_PATH, pbcor_fname)
            if not os.path.exists(pbcor_fpath):
                print(f'[skip] pbcor not found: {pbcor_fname}')
                continue
            print(f'Plotting: snapshot {snapshot} | {field} | axis {axis}')
            plot_three_panel(pbcor_fpath, fit, SKYMODEL_PATH, RESIDUAL_PATH,
                              zoom_factor=4, cmap=CMAP, dpi=DPI, out_dir=OUT_DIR, show=True)
